# Crowd Forecast Model Training

Notebook nay duoc thiet ke cho Google Colab de train baseline crowd forecast model dau tien cho Eco-Travel Coordinator.
Muc tieu la de lam truoc, de mo rong sau, va ho tro schema da thanh pho tren toan Viet Nam.

## 1. Cai dat thu vien

Cell nay cai cac goi can thiet de train, danh gia va export model.

In [ ]:
!pip install -q pandas scikit-learn joblib xgboost

## 2. Cau hinh duong dan

Upload thu muc `eco_travel_coordinator` len Colab hoac mount Google Drive, sau do cap nhat `PROJECT_DIR` neu can.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/content/eco_travel_coordinator')
DATA_DIR = PROJECT_DIR / 'data' / 'training'
MODELS_DIR = PROJECT_DIR / 'models_artifacts'

TRAINING_FILE = DATA_DIR / 'crowd_training_dataset.csv'
CLEANED_OUTPUT = DATA_DIR / 'cleaned_training_data.csv'
MODEL_OUTPUT = MODELS_DIR / 'crowd_forecast_model.pkl'
PREPROCESSOR_OUTPUT = MODELS_DIR / 'crowd_preprocessor.pkl'
METADATA_OUTPUT = MODELS_DIR / 'crowd_model_metadata.json'

PROJECT_DIR.exists(), TRAINING_FILE

## 3. Load dataset

File nay duoc tao tu `scripts/prepare_training_data.py` o local.

In [ ]:
import json
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

df = pd.read_csv(TRAINING_FILE)
print(df.shape)
df.head()

## 4. EDA co ban

Kiem tra city, category va phan bo target truoc khi train.

In [ ]:
print('So attraction:', df['attraction_id'].nunique())
print('So city:', df['city'].nunique())
print('Cities:', sorted(df['city'].dropna().unique().tolist()))
print('Crowd score summary:')
print(df['crowd_score'].describe())
df[['city', 'category', 'weather']].head()

## 5. Clean data va chon feature

Feature bat buoc theo huong mo rong da thanh pho:
- attraction_id
- city
- category
- popularity_score
- estimated_capacity
- hour
- day_of_week
- weather
- holiday_flag
- event_flag
- crowd_score

Feature bo sung hien tai:
- historical_crowd_score
- indoor_outdoor
- temperature
- rain_flag

In [ ]:
target_column = 'crowd_score'
feature_columns = [
    'attraction_id',
    'city',
    'category',
    'popularity_score',
    'estimated_capacity',
    'hour',
    'day_of_week',
    'weather',
    'holiday_flag',
    'event_flag',
    'historical_crowd_score',
    'indoor_outdoor',
    'temperature',
    'rain_flag',
]

model_df = df[feature_columns + [target_column]].copy()
model_df = model_df.dropna(subset=[target_column])
model_df[target_column] = model_df[target_column].clip(0, 100)

categorical_features = ['attraction_id', 'city', 'category', 'day_of_week', 'weather', 'indoor_outdoor']
numeric_features = [column for column in feature_columns if column not in categorical_features]

model_df.to_csv(CLEANED_OUTPUT, index=False)
model_df.head()

## 6. Encode categorical features va chia train/validation

In [ ]:
X = model_df[feature_columns]
y = model_df[target_column]

preprocessor = ColumnTransformer(
    transformers=[
        (
            'categorical',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
        (
            'numeric',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
            ]),
            numeric_features,
        ),
    ]
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

X_train.shape, X_valid.shape

## 7. Train baseline voi RandomForestRegressor

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    (
        'model',
        RandomForestRegressor(
            n_estimators=250,
            max_depth=14,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
    ),
])

rf_pipeline.fit(X_train, y_train)
rf_predictions = rf_pipeline.predict(X_valid)

rf_metrics = {
    'model_name': 'random_forest',
    'mae': float(mean_absolute_error(y_valid, rf_predictions)),
    'rmse': float(mean_squared_error(y_valid, rf_predictions, squared=False)),
    'r2': float(r2_score(y_valid, rf_predictions)),
}
rf_metrics

## 8. Thu nghiem XGBoost neu can

Khong bat buoc. Baseline chinh van la RandomForestRegressor.

In [ ]:
best_pipeline = rf_pipeline
best_metrics = rf_metrics.copy()

try:
    from xgboost import XGBRegressor

    xgb_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        (
            'model',
            XGBRegressor(
                n_estimators=400,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                objective='reg:squarederror',
                random_state=42,
            ),
        ),
    ])

    xgb_pipeline.fit(X_train, y_train)
    xgb_predictions = xgb_pipeline.predict(X_valid)
    xgb_metrics = {
        'model_name': 'xgboost',
        'mae': float(mean_absolute_error(y_valid, xgb_predictions)),
        'rmse': float(mean_squared_error(y_valid, xgb_predictions, squared=False)),
        'r2': float(r2_score(y_valid, xgb_predictions)),
    }

    if xgb_metrics['rmse'] < best_metrics['rmse']:
        best_pipeline = xgb_pipeline
        best_metrics = xgb_metrics

    print(xgb_metrics)
except Exception as exc:
    print('Bo qua XGBoost:', exc)

best_metrics

## 9. Export model, preprocessor va metadata

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

fitted_preprocessor = best_pipeline.named_steps['preprocessor']
fitted_model = best_pipeline.named_steps['model']

joblib.dump(fitted_model, MODEL_OUTPUT)
joblib.dump(fitted_preprocessor, PREPROCESSOR_OUTPUT)

metadata = {
    'model_name': best_metrics['model_name'],
    'target_column': target_column,
    'feature_columns': feature_columns,
    'categorical_features': categorical_features,
    'numeric_features': numeric_features,
    'metrics': best_metrics,
    'training_rows': int(len(model_df)),
    'city_count': int(model_df['city'].nunique()),
}

METADATA_OUTPUT.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved cleaned data to', CLEANED_OUTPUT)
print('Saved model to', MODEL_OUTPUT)
print('Saved preprocessor to', PREPROCESSOR_OUTPUT)
print('Saved metadata to', METADATA_OUTPUT)

## 10. Huong mo rong da thanh pho

- Giu cot `city` va bo sung `district`, `tourism_cluster` khi mo rong du lieu.
- Co the split validation theo city hoac theo thoi gian khi dataset lon hon.
- Co the them feature lich su rolling mean, holiday calendar va event calendar theo dia phuong.
- File export se duoc dat trong `models_artifacts/` de sau nay noi vao app desktop.